# Session 6 — GPT-2 Text Generation
**Task:** Fine-tune GPT-2 on a custom corpus, then generate text with various sampling strategies  
**Model:** `gpt2` (decoder-only, 124M params)  
**Dataset:** WikiText-2 (small Wikipedia subset)  
**Metric:** Perplexity

---
### Key difference from all previous sessions
- GPT-2 = **decoder-only** — no encoder, no cross-attention  
- Trained with **causal language modeling** (CLM): predict next token given previous tokens  
- Input and labels are the **same sequence** — labels are input shifted by one position  
- `generate()` supports: greedy, beam search, top-k, top-p (nucleus), temperature

## Step 1 — Imports & Config

In [ ]:
import os
import math
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import GPT2Tokenizer, GPT2LMHeadModel, get_linear_schedule_with_warmup
from datasets import load_dataset

DEVICE     = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "gpt2"
BLOCK_SIZE = 128    # chunk size for CLM training
BATCH_SIZE = 8
EPOCHS     = 3
LR         = 5e-5
TRAIN_SIZE = 3000
VAL_SIZE   = 300
SAVE_DIR   = "../../models/05_transformers/gpt2_generation"

print(f"Device: {DEVICE}")

## Step 2 — Load & Inspect Dataset

In [ ]:
raw = load_dataset("wikitext", "wikitext-2-raw-v1")
print(raw)

print("\nSample text:")
print(raw["train"][10]["text"][:300])

## Step 3 — Tokenizer
GPT-2 uses BPE (byte pair encoding). No `[CLS]`/`[SEP]` — just raw token ids.  
For CLM training: `input = tokens[:-1]`, `labels = tokens[1:]` — predict next token at every position.

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default

sample = "The quick brown fox"
enc    = tokenizer(sample, return_tensors="pt")
print("Tokens:    ", tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))
print("input_ids: ", enc["input_ids"][0].tolist())
print("\nFor CLM:")
print("  Input  (tokens[:-1]):", tokenizer.convert_ids_to_tokens(enc["input_ids"][0][:-1]))
print("  Labels (tokens[1:]) :", tokenizer.convert_ids_to_tokens(enc["input_ids"][0][1:]))

## Step 4 — Dataset, Model & Training

In [ ]:
class CLMDataset(Dataset):
    def __init__(self, hf_split, tokenizer, block_size=BLOCK_SIZE):
        # Concatenate all text, tokenize once, split into fixed-size blocks
        texts = " ".join([x["text"] for x in hf_split if x["text"].strip()])
        tokens = tokenizer.encode(texts)
        self.examples = [
            torch.tensor(tokens[i: i + block_size], dtype=torch.long)
            for i in range(0, len(tokens) - block_size, block_size)
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        block = self.examples[idx]
        return {"input_ids": block, "labels": block.clone()}   # labels = input (GPT2LMHeadModel handles the shift internally)


train_raw = raw["train"].select(range(TRAIN_SIZE))
val_raw   = raw["validation"].select(range(VAL_SIZE))

train_ds = CLMDataset(train_raw, tokenizer)
val_ds   = CLMDataset(val_raw,   tokenizer)
print(f"Train blocks: {len(train_ds)}  |  Val blocks: {len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE)
print(f"GPT-2 params: {sum(p.numel() for p in model.parameters()):,}")

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        labels    = batch["labels"].to(DEVICE)
        optimizer.zero_grad()
        loss = model(input_ids=input_ids, labels=labels).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            loss = model(input_ids=batch["input_ids"].to(DEVICE), labels=batch["labels"].to(DEVICE)).loss
            total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    return avg_loss, math.exp(avg_loss)   # loss + perplexity


for epoch in range(1, EPOCHS + 1):
    tl = train_epoch(model, train_loader, optimizer, scheduler)
    vl, ppl = evaluate(model, val_loader)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss: {tl:.4f} | val_loss: {vl:.4f} | perplexity: {ppl:.2f}")

## Step 5 — Save & Generation Strategies
Compare greedy, beam search, top-k, and top-p (nucleus) sampling.

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}\n")


def generate(prompt, model, tokenizer, **kwargs):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=60, pad_token_id=tokenizer.eos_token_id, **kwargs)
    return tokenizer.decode(output[0], skip_special_tokens=True)


prompt = "The history of artificial intelligence began"

print("── Greedy (deterministic, repetitive) ──")
print(generate(prompt, model, tokenizer))

print("\n── Beam Search (4 beams) ──")
print(generate(prompt, model, tokenizer, num_beams=4, early_stopping=True))

print("\n── Top-K sampling (k=50) ──")
print(generate(prompt, model, tokenizer, do_sample=True, top_k=50, temperature=0.9))

print("\n── Top-P / Nucleus (p=0.92) ──")
print(generate(prompt, model, tokenizer, do_sample=True, top_p=0.92, temperature=0.9))